# DermaMNIST + HDFS + Spark

The purpose of this notebook is to evaluate a Spark + HDFS data pipeline for the DermaMNIST dataset and compare it with the local baseline.

The dataset is stored in HDFS and accessed using Spark's binary-file reader.

After data is retrieved from HDFS, it is converted to PyTorch tensors and passed to the same CNN architecture used in the baseline experiment.

This notebook does not use distributed neural-network training. PyTorch training is performed by a single process. Distributed training
with PyTorch DDP is evaluated separately in the next experiment.

In [12]:
import io
import os
import sys
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import pyspark

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PySpark:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

Python: 3.11.16 (main, Aug 27 2026, 14:37:14) [GCC 14.3.0]
PyTorch: 2.14.1+cu130
PySpark: 4.1.2
PySpark location: /home/hduser/.conda/envs/cnn/lib/python3.11/site-packages/pyspark/__init__.py


### 1. Reproducibility

We keep it identical to the baseline.

In [2]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

g = torch.Generator()
g.manual_seed(seed)

### 2. Prepare the dataset for HDFS + Spark

The original DermaMNIST dataset is provided as a single `.npz` archive containing all training, validation, and test images and labels.

In the first attempt, the entire `.npz` archive was read by Spark as a single binary object. This required the approximately 100 MB file to be serialized between the JVM and Python processes, resulting in excessive memory usage, and the first row of binary data frame never gets returned.

For the Spark experiment, the archive is therefore unpacked into individual image files organised by dataset split and class.

This representation allows Spark to access and process individual image files rather than transferring one large binary object.

The original `.npz` dataset remains unchanged and is used as the source for this preparation step.

In [11]:
# original local DermaMNIST dataset
npz_path = "/home/hduser/.medmnist/dermamnist_64.npz"

# local directory that will be uploaded to HDFS
output_dir = "/home/hduser/Desktop/CA1/dermamnist_spark"

os.makedirs(output_dir, exist_ok=True)

# load the original dataset
with np.load(npz_path) as data:
    datasets = {
        "train": (data["train_images"], data["train_labels"].flatten()),
        "val":   (data["val_images"], data["val_labels"].flatten()),
        "test":  (data["test_images"], data["test_labels"].flatten())
    }

# convert each image to PNG and organise by split/class
for split, (images, labels) in datasets.items():

    for class_id in np.unique(labels):
        class_dir = os.path.join(
            output_dir,
            split,
            str(class_id)
        )
        os.makedirs(class_dir, exist_ok=True)

    for i, (image, label) in enumerate(zip(images, labels)):
        image_path = os.path.join(
            output_dir,
            split,
            str(label),
            f"{i:05d}.png"
        )

        Image.fromarray(image).save(image_path)

print(f"Dataset prepared in: {output_dir}")

Dataset prepared in: /home/hduser/Desktop/CA1/dermamnist_spark


In [13]:
# check what we got
for split in ["train", "val", "test"]:
    count = len(list(
        Path(output_dir, split).rglob("*.png")
    ))
    print(f"{split}: {count} images")

train: 7007 images
val: 1003 images
test: 2005 images


#### Store the prepared dataset in HDFS

The prepared image dataset is now copied to HDFS. This upload is treated as experiment setup and is not included in the pipeline timing, just as the original DermaMNIST download was not included in the baseline timing.

### 3. Start Spark

In [14]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DermaMNIST_Spark_HDFS")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

Spark version: 4.1.2
Master: local[*]


In [15]:
# that Spark is connected to the Hadoop filesystem we're using
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

print("Default filesystem:", hadoop_conf.get("fs.defaultFS"))

Default filesystem: hdfs://localhost:9000


### 4. Access DermaMNIST from HDFS

The DermaMNIST dataset was downloaded before the experiment and stored in HDFS at `/ca1/dermamnist/dermamnist_64.npz`.

The download and HDFS upload are treated as dataset setup and are not included in the pipeline timing. This allows the timing comparison to focus on the data-processing and model pipeline itself.

In [16]:
hdfs_path = "/ca1/dermamnist_spark/*"

In [17]:
# read the .npz using Spark
binary_df = (spark.read.format("binaryFile").load(hdfs_path))

binary_df.select("path", "length").show(truncate=False)

26/10/05 10:04:48 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /ca1/dermamnist_spark/*.
java.io.FileNotFoundException: File does not exist: /ca1/dermamnist_spark/*
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analys

+----+------+
|path|length|
+----+------+
+----+------+

